<a href="https://colab.research.google.com/github/yuniecorn-dev/Numerai_ESAA/blob/main/xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install xgboost optuna -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import gc

FILE_PATH = "/content/drive/MyDrive/26-1/ESAA/[ESAA] YB 1조/컨퍼런스/pca500_lag1_lag2"
RANDOM_STATE = 42

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 33.6 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
import pyarrow.parquet as pq

schema = pq.ParquetFile(FILE_PATH).schema_arrow
all_cols = schema.names

base_cols = [c for c in all_cols if c.startswith('pca_') and not c.endswith(('_lag1', '_lag2'))]
lag1_cols = [c for c in all_cols if c.endswith('_lag1')]
lag2_cols = [c for c in all_cols if c.endswith('_lag2')]
FEATURES  = base_cols + lag1_cols + lag2_cols

print(f"base: {len(base_cols)}, lag1: {len(lag1_cols)}, lag2: {len(lag2_cols)}, total: {len(FEATURES)}")

use_cols = FEATURES + ['era', 'target']
df = pd.read_parquet(FILE_PATH, columns=use_cols)
df[FEATURES] = df[FEATURES].astype('float32').fillna(0)
df['era'] = df['era'].astype(int)

print(df.shape)
gc.collect()

base: 500, lag1: 500, lag2: 500, total: 1500
(587583, 1502)


30

In [4]:
unique_eras = sorted(df['era'].unique())
split_point = int(len(unique_eras) * 0.8)
train_eras = unique_eras[:split_point]
valid_eras = unique_eras[split_point:]

train_df = df[df['era'].isin(train_eras)].reset_index(drop=True)
val_df   = df[df['era'].isin(valid_eras)].reset_index(drop=True)

print(f"train era: {len(train_eras)}개, valid era: {len(valid_eras)}개")
print(f"train shape: {train_df.shape}, val shape: {val_df.shape}")

train era: 92개, valid era: 23개
train shape: (461940, 1502), val shape: (125643, 1502)


In [5]:
def numerai_corr_arrays(era_arr, target_arr, pred_arr):
    df_eval = pd.DataFrame({"era": era_arr, "target": target_arr, "prediction": pred_arr})
    def era_corr(sub):
        return np.corrcoef(sub["prediction"].rank(pct=True), sub["target"])[0, 1]
    return df_eval.groupby("era").apply(era_corr, include_groups=False)

In [6]:
model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=3.0,
    eval_metric="rmse",
    random_state=RANDOM_STATE,
    device="cuda",
    early_stopping_rounds=100,
)

model.fit(
    train_df[FEATURES], train_df["target"],
    eval_set=[(val_df[FEATURES], val_df["target"])],
    verbose=100
)

[0]	validation_0-rmse:0.22311
[100]	validation_0-rmse:0.22301
[200]	validation_0-rmse:0.22299
[300]	validation_0-rmse:0.22295
[400]	validation_0-rmse:0.22293
[500]	validation_0-rmse:0.22292
[600]	validation_0-rmse:0.22292
[617]	validation_0-rmse:0.22292


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=100,
             enable_categorical=True, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.02, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2000,
             n_jobs=None, num_parallel_tree=None, ...)

In [7]:
val_preds = model.predict(val_df[FEATURES])
per_era_corr = numerai_corr_arrays(val_df["era"].values, val_df["target"].values, val_preds)

mean_corr = per_era_corr.mean()
std_corr  = per_era_corr.std()
sharpe    = mean_corr / std_corr

print(f"Validation mean corr: {mean_corr:.4f}")
print(f"Validation std corr:  {std_corr:.4f}")
print(f"Validation sharpe:    {sharpe:.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [23:06:27] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Validation mean corr: 0.0404
Validation std corr:  0.0332
Validation sharpe:    1.2152


In [9]:
result_df = pd.DataFrame({
    "era": val_df["era"].values,
    "prediction": val_preds
})
result_df.to_csv("/content/drive/MyDrive/26-1/ESAA/[ESAA] YB 1조/컨퍼런스/베이스모델 노트북/seoyun_xgboost_val_predictions.csv", index=False)
model.save_model("/content/drive/MyDrive/26-1/ESAA/[ESAA] YB 1조/컨퍼런스/베이스모델 노트북/xgboost_base_v1.json")
print("저장 완료!")

저장 완료!


In [10]:
def objective(trial):
    params = {
        "n_estimators": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "eval_metric": "rmse",
        "random_state": RANDOM_STATE,
        "device": "cuda",
        "early_stopping_rounds": 100,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(
        train_df[FEATURES], train_df["target"],
        eval_set=[(val_df[FEATURES], val_df["target"])],
        verbose=False
    )
    preds = model.predict(val_df[FEATURES])
    per_era_corr = numerai_corr_arrays(val_df["era"].values, val_df["target"].values, preds)
    sharpe = per_era_corr.mean() / per_era_corr.std()
    trial.set_user_attr("mean_corr", float(per_era_corr.mean()))
    trial.set_user_attr("std_corr", float(per_era_corr.std()))
    return sharpe

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"=== 1차 탐색 Best ===")
print(f"  sharpe: {study.best_trial.value:.4f}")
print(f"  params: {study.best_trial.params}")

  0%|          | 0/30 [00:00<?, ?it/s]

=== 1차 탐색 Best ===
  sharpe: 2.6532
  params: {'learning_rate': 0.01683695685806219, 'max_depth': 3, 'subsample': 0.6842255443980394, 'colsample_bytree': 0.9160641130149365, 'reg_alpha': 0.27188026107491425, 'reg_lambda': 6.848672546896612, 'min_child_weight': 7}


In [11]:
results = []
for t in study.trials:
    row = {"trial": t.number, "sharpe": t.value, **t.user_attrs, **t.params}
    results.append(row)

results_df = pd.DataFrame(results).sort_values("sharpe", ascending=False)
results_df.to_csv("/content/drive/MyDrive/26-1/ESAA/[ESAA] YB 1조/컨퍼런스/베이스모델 노트북/xgboost_tuning_results.csv", index=False)
print(results_df.head(10))

    trial    sharpe  mean_corr  std_corr  learning_rate  max_depth  subsample  \
17     17  2.653156   0.034792  0.013114       0.016837          3   0.684226   
19     19  2.519040   0.034679  0.013767       0.016857          3   0.690393   
12     12  2.441112   0.030834  0.012631       0.035330          3   0.739562   
11     11  2.409190   0.036313  0.015073       0.040285          3   0.778802   
24     24  2.354847   0.032217  0.013681       0.023705          3   0.586524   
14     14  2.295613   0.036608  0.015947       0.027398          3   0.760801   
22     22  2.269502   0.032797  0.014451       0.013576          3   0.665288   
28     28  2.250388   0.036273  0.016118       0.014506          3   0.714200   
13     13  2.240121   0.031891  0.014236       0.030180          3   0.736866   
26     26  2.212125   0.032058  0.014492       0.019649          3   0.652430   

    colsample_bytree  reg_alpha  reg_lambda  min_child_weight  
17          0.916064   0.271880    6.848673 

In [12]:
best_params = study.best_trial.params
best_params.update({
    "n_estimators": 2000,
    "eval_metric": "rmse",
    "random_state": RANDOM_STATE,
    "device": "cuda",
    "early_stopping_rounds": 100,
})

final_model = xgb.XGBRegressor(**best_params)
final_model.fit(
    train_df[FEATURES], train_df["target"],
    eval_set=[(val_df[FEATURES], val_df["target"])],
    verbose=100
)

final_preds = final_model.predict(val_df[FEATURES])
per_era_corr_final = numerai_corr_arrays(val_df["era"].values, val_df["target"].values, final_preds)

print(f"[최종 모델] mean corr: {per_era_corr_final.mean():.4f}")
print(f"[최종 모델] std corr:  {per_era_corr_final.std():.4f}")
print(f"[최종 모델] sharpe:    {per_era_corr_final.mean()/per_era_corr_final.std():.4f}")

[0]	validation_0-rmse:0.22311
[100]	validation_0-rmse:0.22306
[200]	validation_0-rmse:0.22304
[300]	validation_0-rmse:0.22302
[400]	validation_0-rmse:0.22301
[500]	validation_0-rmse:0.22299
[600]	validation_0-rmse:0.22298
[700]	validation_0-rmse:0.22298
[800]	validation_0-rmse:0.22297
[900]	validation_0-rmse:0.22297
[925]	validation_0-rmse:0.22297
[최종 모델] mean corr: 0.0348
[최종 모델] std corr:  0.0131
[최종 모델] sharpe:    2.6532


In [13]:
best_p = study.best_trial.params

def objective_v2(trial):
    params = {
        "n_estimators": 2000,
        "learning_rate": trial.suggest_float("learning_rate",
            best_p["learning_rate"]*0.5, best_p["learning_rate"]*2.0, log=True),
        "max_depth": trial.suggest_int("max_depth",
            max(2, best_p["max_depth"]-1), best_p["max_depth"]+1),
        "subsample": trial.suggest_float("subsample",
            max(0.5, best_p["subsample"]-0.15), min(1.0, best_p["subsample"]+0.15)),
        "colsample_bytree": trial.suggest_float("colsample_bytree",
            max(0.4, best_p["colsample_bytree"]-0.15), min(1.0, best_p["colsample_bytree"]+0.15)),
        "reg_alpha": trial.suggest_float("reg_alpha",
            best_p["reg_alpha"]*0.3, best_p["reg_alpha"]*3.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda",
            best_p["reg_lambda"]*0.3, best_p["reg_lambda"]*3.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight",
            max(1, best_p["min_child_weight"]-2), best_p["min_child_weight"]+2),
        "eval_metric": "rmse",
        "random_state": RANDOM_STATE,
        "device": "cuda",
        "early_stopping_rounds": 100,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(
        train_df[FEATURES], train_df["target"],
        eval_set=[(val_df[FEATURES], val_df["target"])],
        verbose=False
    )
    preds = model.predict(val_df[FEATURES])
    per_era_corr = numerai_corr_arrays(val_df["era"].values, val_df["target"].values, preds)
    return per_era_corr.mean() / per_era_corr.std()

study_v2 = optuna.create_study(direction="maximize")
study_v2.optimize(objective_v2, n_trials=30, show_progress_bar=True)

print(f"=== 2차 탐색 Best ===")
print(f"  sharpe: {study_v2.best_trial.value:.4f}")
print(f"\n=== 1차 vs 2차 비교 ===")
print(f"1차 best sharpe: {study.best_trial.value:.4f}")
print(f"2차 best sharpe: {study_v2.best_trial.value:.4f}")

  0%|          | 0/30 [00:00<?, ?it/s]

=== 2차 탐색 Best ===
  sharpe: 2.4645

=== 1차 vs 2차 비교 ===
1차 best sharpe: 2.6532
2차 best sharpe: 2.4645


In [14]:
if study_v2.best_trial.value > study.best_trial.value:
    print("2차 탐색이 더 좋음")
    best_overall_params = study_v2.best_trial.params
else:
    print("1차 탐색이 더 좋음")
    best_overall_params = study.best_trial.params

best_overall_params.update({
    "n_estimators": 2000,
    "eval_metric": "rmse",
    "random_state": RANDOM_STATE,
    "device": "cuda",
    "early_stopping_rounds": 100,
})

final_model_v2 = xgb.XGBRegressor(**best_overall_params)
final_model_v2.fit(
    train_df[FEATURES], train_df["target"],
    eval_set=[(val_df[FEATURES], val_df["target"])],
    verbose=100
)

final_preds_v2 = final_model_v2.predict(val_df[FEATURES])
per_era_corr_v2 = numerai_corr_arrays(val_df["era"].values, val_df["target"].values, final_preds_v2)

print(f"[최종] mean corr: {per_era_corr_v2.mean():.4f}")
print(f"[최종] std corr:  {per_era_corr_v2.std():.4f}")
print(f"[최종] sharpe:    {per_era_corr_v2.mean()/per_era_corr_v2.std():.4f}")

final_model_v2.save_model("/content/drive/MyDrive/26-1/ESAA/[ESAA] YB 1조/컨퍼런스/베이스모델 노트북/xgboost_best_v2.json")
pd.DataFrame({
    "era": val_df["era"].values,
    "prediction": final_preds_v2
}).to_csv("/content/drive/MyDrive/26-1/ESAA/[ESAA] YB 1조/컨퍼런스/베이스모델 노트북/seoyun_xgboost_val_predictions.csv", index=False)
print("최종 모델(v2) & 예측값 저장 완료!")

1차 탐색이 더 좋음
[0]	validation_0-rmse:0.22311
[100]	validation_0-rmse:0.22306
[200]	validation_0-rmse:0.22304
[300]	validation_0-rmse:0.22302
[400]	validation_0-rmse:0.22301
[500]	validation_0-rmse:0.22299
[600]	validation_0-rmse:0.22298
[700]	validation_0-rmse:0.22298
[800]	validation_0-rmse:0.22297
[900]	validation_0-rmse:0.22297
[925]	validation_0-rmse:0.22297
[최종] mean corr: 0.0348
[최종] std corr:  0.0131
[최종] sharpe:    2.6532
최종 모델(v2) & 예측값 저장 완료!
